《Think Stats》 3판은 [Bookshop.org](https://bookshop.org/a/98697/9781098190255)와 [Amazon](https://amzn.to/42lmxwu)에서 구할 수 있습니다. 두 링크는 원저자의 제휴 링크입니다. 책이 도움이 되었다면 Green Tea Press의 [무료 공개 교육 자료 제작을 후원](https://greenteapress.com/support/?book=ts3)하는 것도 생각해 주세요.

# 다중회귀

앞 장의 선형 최소제곱은 **회귀**의 한 예입니다. 회귀는 예측할 변수인 **반응변수** 또는 종속변수와, 예측에 사용하는 **설명변수** 또는 독립변수 사이의 관계를 모형으로 나타내는 문제입니다.

앞 장처럼 반응변수와 설명변수가 하나씩이면 **단순회귀**입니다. 이 장에서는 반응변수 하나를 여러 설명변수로 예측하는 **다중회귀**로 확장합니다. 반응변수까지 여러 개인 다변량 회귀는 이 책에서 다루지 않습니다.

[원문 노트북을 Colab에서 실행하기](https://colab.research.google.com/github/AllenDowney/ThinkStats/blob/30ecbdcf05723f1190cae2aeaff5848f59f8ca26/nb/chap11.ipynb).

In [1]:
from os.path import basename, exists


def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + local)


download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")

In [2]:
try:
    import empiricaldist
except ImportError:
    %pip install empiricaldist

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thinkstats import decorate

## statsmodels

앞 장의 SciPy `linregress`는 단순회귀를 계산합니다. 다중회귀에는 여러 회귀와 분석 기능을 제공하는 `statsmodels`를 사용하겠습니다. 먼저 펭귄 자료를 계속 살펴보겠습니다.

자료는 Horst, Hill, Gorman(2020), *palmerpenguins*, [DOI: 10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218)에서 제공합니다. 원래 연구는 Gorman, Williams, Fraser(2014), *PLOS ONE* 9(3), e90081, [DOI: 10.1371/journal.pone.0090081](https://doi.org/10.1371/journal.pone.0090081)입니다.

In [4]:
download(
    "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/c19a904462482430170bfe2c718775ddb7dbb885/inst/extdata/penguins_raw.csv"
)

공백 없는 열 이름을 쓰면 수식 API가 편리합니다. 다음 사전으로 이름을 바꾸겠습니다.

In [5]:
columns = {
    "Body Mass (g)": "mass",
    "Flipper Length (mm)": "flipper_length",
    "Culmen Length (mm)": "culmen_length",
    "Culmen Depth (mm)": "culmen_depth",
}

자료를 읽고 체중이 빠진 행을 제외한 뒤 열 이름을 바꿉니다.

In [6]:
penguins = (
    pd.read_csv("penguins_raw.csv")
    .dropna(subset=["Body Mass (g)"])
    .rename(columns=columns)
)
penguins.shape

세 종 가운데 아델리펭귄만 선택합니다.

In [7]:
adelie = penguins.query('Species.str.startswith("Adelie")').copy()
len(adelie)

앞 장에서는 날개 길이로 체중을 설명하는 최소제곱 직선을 구했습니다.

In [8]:
flipper_length = adelie["flipper_length"]
body_mass = adelie["mass"]

`linregress`를 쓴 코드를 다시 보겠습니다.

In [9]:
from scipy.stats import linregress

result_linregress = linregress(flipper_length, body_mass)
result_linregress.intercept, result_linregress.slope

statsmodels의 두 인터페이스 중 Patsy 수식 언어를 사용하는 formula API를 쓰겠습니다. 다음 문자열은 반응변수 `mass`가 설명변수 `flipper_length`의 선형함수라는 뜻입니다.

In [10]:
formula = "mass ~ flipper_length"

수식과 자료를 `ols`에 전달합니다.

In [11]:
import statsmodels.formula.api as smf

model = smf.ols(formula, data=adelie)
type(model)

`ols`는 ordinary least squares, 즉 **보통최소제곱**의 줄임말입니다. 반환된 `OLS` 객체는 반응변수가 설명변수의 선형함수라는 모형을 나타냅니다. `fit`으로 계수를 추정하면 결과를 담은 `RegressionResults` 객체가 나옵니다.

In [12]:
result_ols = model.fit()

In [13]:
# Technically it's a RegressionResultsWrapper

type(result_ols)

결과 객체의 정보가 많으므로, 원문에서는 `display_summary`로 지금 필요한 부분만 표시합니다.

In [14]:
from thinkstats import display_summary

display_summary(result_ols)

첫 번째 열은 모형의 **계수**인 절편과 기울기입니다. `linregress`와 같은 값인지 확인하겠습니다.

In [15]:
result_linregress.intercept, result_linregress.slope

다음 열은 계수들의 표준오차입니다. 이 값도 `linregress`의 결과와 같습니다.

In [16]:
result_linregress.intercept_stderr, result_linregress.stderr,

다음에는 p값 계산에 쓰는 t통계량이 있고, `P>|t|` 열에 양측 p값이 있습니다. 표에 0으로 반올림된 날개 길이의 p값을 더 자세히 출력해 보겠습니다.

In [17]:
result_ols.pvalues["flipper_length"]

`linregress`의 p값과도 비교합니다.

In [18]:
result_linregress.pvalue

p값은 매우 작습니다. 모형의 가정 아래 체중과 날개 길이의 선형 관계가 없다면, 관측한 t통계량의 절댓값 이상으로 극단적인 결과가 나오기 어렵다는 뜻입니다. 마지막 두 열은 계수의 95% 신뢰구간이며, 기울기의 구간은 약 [22.8, 42.9]g/mm입니다.

마지막 줄의 $R^2$는 약 0.22입니다. 평균 체중만 사용할 때보다 훈련 자료의 MSE를 약 22% 줄입니다. 단순 최소제곱의 `rsquared`를 `linregress`의 상관계수 제곱과 비교하겠습니다.

In [19]:
result_ols.rsquared, result_linregress.rvalue**2

부동소수점 근사 차이를 빼면 같습니다. 다중회귀에 들어가기 전에, 이번에는 부리 길이 `culmen_length`만 설명변수로 사용하겠습니다.

In [20]:
formula = "mass ~ culmen_length"
result = smf.ols(formula, data=adelie).fit()
display_summary(result)

부리 길이의 기울기도 p값이 작습니다. 절편의 p값이 커도, 이 문제에서는 절편이 0인지가 관심사가 아니므로 곧바로 문제라는 뜻은 아닙니다. 여기서 절편은 부리 길이가 0mm인 경우를 가리키므로 실제 관측 범위의 관계를 설명하는 데 중요하지 않습니다. 이 모형의 $R^2$는 약 0.30으로 날개 길이 모형의 약 0.22보다 큽니다. 두 측정값을 함께 쓰면 어떻게 될까요?

## 다중회귀로 확장하기

체중을 날개 길이와 부리 길이 두 설명변수의 선형함수로 나타내는 수식입니다.

In [21]:
formula = "mass ~ flipper_length + culmen_length"

이 모형을 자료에 맞춘 결과입니다.

In [22]:
result = smf.ols(formula, data=adelie).fit()
display_summary(result)

절편 하나와 기울기 둘이 나옵니다. 날개 길이 계수 약 22.7g/mm는 **부리 길이가 같을 때** 날개가 1mm 긴 펭귄의 예측 체중이 약 22.7g 높다는 뜻입니다. 부리 길이 계수 약 76.3g/mm도 **날개 길이가 같을 때**의 관계입니다.

두 기울기의 p값은 작습니다. $R^2$는 약 0.39로, 부리 길이만 쓴 0.30이나 날개 길이만 쓴 0.22보다 큽니다. 하지만 0.22+0.30=0.52만큼 높아지지는 않습니다. 두 설명변수가 서로 상관되어 있기 때문입니다.

In [23]:
from thinkstats import corrcoef

corrcoef(adelie, "flipper_length", "culmen_length")

날개가 긴 펭귄은 부리도 긴 경향이 있습니다. 두 변수는 반응변수에 관한 정보를 일부 공유합니다. 변수를 추가할 때 $R^2$의 증가는 새 변수가 더해 준 정보만 반영합니다. 부리 깊이까지 추가해 보겠습니다.

In [24]:
formula = "mass ~ flipper_length + culmen_length + culmen_depth"
result = smf.ols(formula, data=adelie).fit()
display_summary(result)

계수는 네 개가 됩니다. 각 설명변수의 p값이 작고 $R^2$는 약 0.51로 증가합니다. 다만 부리 깊이도 나머지 측정값과 상관되어 있어 증가량은 단순히 더해지지 않습니다.

In [25]:
[
    corrcoef(adelie, "culmen_depth", "flipper_length"),
    corrcoef(adelie, "culmen_depth", "culmen_length"),
]

이처럼 여러 설명변수를 결합해 예측을 개선하는 것이 다중회귀의 한 용도입니다. 또 다른 용도는 다른 변수의 기여를 고려한 상태에서 관심 변수의 관계를 추정하는 것입니다.

> 번역자 주: 여기서 $R^2$ 비교는 같은 훈련 자료에 대한 비교입니다. 설명변수를 추가하면 보통최소제곱의 훈련 $R^2$는 내려가지 않지만, 새 자료 예측이 반드시 좋아지지는 않습니다. 설명변수라는 이름도 그 변수가 원인임을 보장하지 않습니다.

## 통제변수

[4장](/posts/statistics/think-stats-04#cdf-비교하기)에서는 첫아이가 평균적으로 조금 가벼웠고, [9장](/posts/statistics/think-stats-09#상관-검정하기)에서는 나이가 많은 산모의 아기가 조금 무거운 경향을 보았습니다.

첫아이의 산모가 더 젊다면 두 결과가 연결될 수 있습니다. 다중회귀로 산모 나이를 고려한 상태에서 첫아이와 다른 아이의 체중 차이를 추정해 보겠습니다. NSFG 자료를 불러오는 코드는 노트북에 있습니다.

1장에서와 같은 NSFG 자료를 읽고 정제합니다.

In [26]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/nsfg.py")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dct")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dat.gz")

In [27]:
try:
    import statadict
except ImportError:
    %pip install statadict

생아 출산을 고른 뒤 첫아이와 다른 아이로 나눕니다.

In [28]:
from nsfg import get_nsfg_groups

live, firsts, others = get_nsfg_groups()

출생 체중·출생 순서·산모 나이가 모두 유효한 행만 선택합니다.

In [29]:
valid = live.dropna(subset=["agepreg", "birthord", "totalwgt_lb"]).copy()

산모 나이와 체중의 관계를 선형으로 가정하고 회귀하겠습니다.

In [30]:
formula = "totalwgt_lb ~ agepreg"
result_age = smf.ols(formula, data=valid).fit()
display_summary(result_age)

기울기는 약 0.0175lb/년입니다. 나이가 10년 다른 두 산모의 아기는 평균적으로 약 0.175lb 차이가 난다고 예측합니다. 기울기는 작아도 p값은 작습니다. 반면 $R^2$는 아주 작아 나이를 알아도 개별 아기의 체중 예측은 거의 좋아지지 않습니다. 작은 p값과 작은 예측력이 함께 나오는 것은 모순이 아닙니다. 그림으로 확인하기 위해 두 열을 고르겠습니다.

In [31]:
totalwgt = valid["totalwgt_lb"]
agepreg = valid["agepreg"]

계수를 꺼내 직접 계산하지 않고 결과 객체의 `predict`를 사용할 수 있습니다. 먼저 나이의 범위를 만듭니다.

In [32]:
agepreg_range = np.linspace(agepreg.min(), agepreg.max())

`predict`에 전달할 설명변수 값들을 `DataFrame`에 담습니다.

In [33]:
df = pd.DataFrame({"agepreg": agepreg_range})

열 이름이 모형의 설명변수 이름과 같아야 합니다. 이 표를 `predict`에 전달합니다.

In [34]:
fit_ys = result_age.predict(df)

결과인 예측 `Series`를 산점도와 함께 그립니다.

In [35]:
plt.scatter(agepreg, totalwgt, marker=".", alpha=0.1, s=5)
plt.plot(agepreg_range, fit_ys, color="C1", label="linear model")

decorate(xlabel="Maternal age", ylabel="Birth weight (pounds)")

나이에 따른 평균의 차이보다 각 나이에서의 체중 차이가 훨씬 큽니다. 이번에는 첫아이 여부와 체중의 관계를 보겠습니다. 첫아이이면 `True`, 아니면 `False`인 `is_first` 열을 만듭니다.

In [36]:
valid["is_first"] = valid["birthord"] == 1

In [37]:
from thinkstats import value_counts

# check the results
value_counts(valid["is_first"])

반응변수는 `totalwgt_lb`, 설명변수는 `is_first`입니다. Patsy의 `C(...)`는 측정값이 아니라 집단을 나타내는 **범주형 변수**로 취급하라는 뜻입니다.

In [38]:
formula = "totalwgt_lb ~ C(is_first)"

모형을 적합하고 결과를 봅니다.

In [39]:
result_first = smf.ols(formula, data=valid).fit()
display_summary(result_first)

`C(is_first)[T.True]`는 첫아이 집단의 계수입니다. `[T.True]`는 기준범주와 비교한 `True` 범주의 차이를 나타내는 [처리 부호화(treatment coding)](https://www.statsmodels.org/stable/contrasts.html#treatment-dummy-coding) 표기입니다. 기준집단은 다른 아이들입니다. 어느 쪽을 기준으로 삼는지는 선택이지만, 계수의 부호를 읽으려면 알아야 합니다. 절편 약 7.3lb는 기준집단의 평균이고, 첫아이의 계수 약 −0.12lb는 그보다 가볍다는 뜻입니다. 집단 평균으로 직접 확인하겠습니다.

In [40]:
others["totalwgt_lb"].mean()

In [41]:
diff_weight = firsts["totalwgt_lb"].mean() - others["totalwgt_lb"].mean()
diff_weight

첫아이 계수의 p값은 작지만 $R^2$도 작습니다. 집단 평균 차이는 검출되어도 개별 체중을 알아맞히는 데에는 큰 도움이 되지 않습니다. 첫아이 산모는 다른 산모보다 평균 약 3.6년 젊으므로, 이것이 체중 차이를 얼마나 설명할지 보겠습니다.

In [42]:
diff_age = firsts["agepreg"].mean() - others["agepreg"].mean()
diff_age

나이에 따른 체중 기울기는 약 0.0175lb/년입니다.

In [43]:
slope = result_age.params["agepreg"]
slope

나이 차이와 기울기를 곱하면 나이 차이에 대응하는 예측 체중 차이가 나옵니다.

In [44]:
slope * diff_age

약 0.063lb로 관측된 집단 차이의 절반 정도입니다. 산모 나이가 일부를 설명할 수 있어 보입니다. 두 설명변수를 같은 회귀에 넣겠습니다.

In [45]:
formula = "totalwgt_lb ~ agepreg + C(is_first)"
result = smf.ols(formula, data=valid).fit()
display_summary(result)

첫아이 계수는 약 −0.0698lb입니다. 산모 나이의 선형 관계를 함께 고려하면 조정 전 차이의 약 절반이 남습니다. p값은 약 0.025입니다. 관습적인 5% 기준으로는 유의하지만, 그 수치만으로 해석을 확정할 일은 아닙니다.

이 모형은 산모 나이를 **통제한다**고 말합니다. 다만 나이와 체중의 관계가 직선이라는 가정이 들어 있습니다. 그 가정을 확인해 보겠습니다.

> 번역자 주: ‘통제’는 지정한 모형 안에서 관계를 조정했다는 뜻입니다. 관찰 자료에서 빠진 교란변수나 모형 오류까지 없앴다는 뜻은 아닙니다. 이 예제의 기본 표준오차에는 복합표본 설계나 같은 산모의 반복 관측에 대한 조정도 들어 있지 않습니다.

## 비선형 관계

나이의 기여가 곡선인지 보기 위해 산모 나이의 제곱인 `agepreg2` 열을 만듭니다.

In [46]:
valid["agepreg2"] = valid["agepreg"] ** 2

나이의 일차항과 이차항을 함께 넣은 모형을 정의합니다.

In [47]:
formula = "totalwgt_lb ~ agepreg + agepreg2"

같은 방법으로 적합합니다.

In [48]:
result_age2 = smf.ols(formula, data=valid).fit()
display_summary(result_age2)

이차항의 p값은 작고, $R^2$는 선형 모형의 약 0.0047에서 약 0.0072로 증가합니다. 나이와 나이 제곱의 계수를 추정하면 나이에 대한 포물선을 맞추는 셈입니다. 이를 그리기 위해 두 설명변수 값을 담은 표를 만듭니다.

In [49]:
df = pd.DataFrame({"agepreg": agepreg_range})
df["agepreg2"] = df["agepreg"] ** 2

이 표를 `predict`에 전달하면 예측값 `Series`가 나옵니다.

In [50]:
fit_ys = result_age2.predict(df)

예측 포물선과 원자료를 함께 그립니다.

In [51]:
plt.scatter(agepreg, totalwgt, marker=".", alpha=0.1, s=5)
plt.plot(agepreg_range, fit_ys, color="C1", label="quadratic model")

decorate(xlabel="Maternal age", ylabel="Birth weight (pounds)")

휘어짐은 작지만 아주 젊거나 나이가 많은 산모의 아기가 조금 가볍고, 가운데 나이대에서 조금 무거운 패턴을 시사합니다. 이 관계를 고려한 뒤 첫아이 효과를 다시 추정하겠습니다.

In [52]:
formula = "totalwgt_lb ~ agepreg + agepreg2 + C(is_first)"
result = smf.ols(formula, data=valid).fit()
display_summary(result)

첫아이의 체중 차이 추정치는 약 −0.0504lb로, 차이의 절댓값이 더 작아집니다. p값은 약 0.109입니다. 이 모형에서는 남은 차이가 우연한 변동으로도 나타날 만합니다. 원저자는 산모 나이가 관측 차이의 일부, 어쩌면 전부를 설명할 수 있다고 해석합니다.

> 번역자 주: p값이 0.05보다 커졌다는 사실이 ‘차이가 정확히 0’이거나 ‘나이가 인과적으로 전부 설명한다’는 증명은 아닙니다. 추정 계수·구간·모형 가정을 함께 보아야 합니다.

## 로지스틱 회귀

선형회귀에서는 반응변수의 기대값을 설명변수의 가중합과 절편으로 나타냅니다. 체중 같은 연속량에는 유용하지만, 횟수나 범주에는 다른 모형이 필요할 수 있습니다. **일반화선형모형**(GLM)에는 다음과 같은 선택이 있습니다.

- 횟수: 포아송 회귀.
- 두 범주: 로지스틱 회귀.
- 여러 범주: 다항 로지스틱 회귀.
- 순서가 있는 여러 범주: 순서형 로지스틱 회귀.

이 책에서는 널리 쓰이는 **로지스틱 회귀**를 살펴봅니다. 체중과 다른 측정값으로 펭귄의 성별을 구별할 수 있을까요? statsmodels의 `logit`을 사용하기 전에 반응변수를 0과 1로 바꾸어야 합니다. 이름은 로지스틱 회귀에 쓰이는 logit 함수에서 왔습니다.

> 번역자 주: 원문은 `Sex == "MALE"`가 아닌 값을 모두 0으로 바꿔 성별 누락도 암컷으로 취급합니다. 여기서는 먼저 `MALE`·`FEMALE`이 확인된 사례만 고르도록 코드와 뒤의 계산을 고쳤습니다. 아델리펭귄은 146마리, 젠투펭귄은 119마리입니다. 원문의 유사 결정계수 0.526·0.662·0.848과 아래 수치가 다른 이유입니다.

In [ ]:
adelie = adelie.loc[adelie["Sex"].isin(["MALE", "FEMALE"])].copy()
adelie["y"] = (adelie["Sex"] == "MALE").astype(int)

In [54]:
adelie["y"].value_counts()

먼저 체중 `mass`만으로 반응변수 `y`를 설명하겠습니다. `disp=False`는 최적화 과정의 진행 메시지를 생략합니다.

In [55]:
model = smf.logit("y ~ mass", data=adelie)
result = model.fit(disp=False)

적합 결과를 확인합니다.

In [56]:
display_summary(result)

선형회귀의 $R^2$는 로지스틱 회귀에 그대로 적용되지 않습니다. 대신 **유사 결정계수**를 사용합니다. 여기서 출력하는 값은 [맥패든의 유사 결정계수](https://www.statsmodels.org/stable/generated/statsmodels.discrete.discrete_model.LogitResults.prsquared.html)이며 약 0.562입니다. 체중 계수가 양수이므로 무거운 펭귄일수록 수컷일 예측 확률이 높습니다. 계수를 직접 읽기보다, 여러 체중의 예측값을 계산해 보겠습니다.

In [57]:
mass = adelie["mass"]
mass_range = np.linspace(mass.min(), mass.max())
df = pd.DataFrame({"mass": mass_range})
fit_ys = result.predict(df)

예측값은 해당 체중의 펭귄이 수컷일 확률입니다. 그려 보겠습니다.

In [58]:
plt.plot(mass_range, fit_ys)

decorate(xlabel="Mass (g)", ylabel="Prob(male)")

가장 가벼운 아델리펭귄은 암컷일 가능성이 높고, 가장 무거운 개체는 수컷일 가능성이 높습니다. 약 3680g에서는 두 가능성이 비슷합니다. 다른 측정값들도 설명변수로 추가하겠습니다.

In [59]:
formula = "y ~ mass + flipper_length + culmen_length + culmen_depth"
model = smf.logit(formula, data=adelie)
result = model.fit(disp=False)
display_summary(result)

유사 결정계수는 약 0.735로 증가합니다. 부리 길이와 깊이의 p값은 작지만, 날개 길이의 p값은 큽니다. 이 모형에서는 체중과 부리 크기를 고려한 뒤 날개 길이가 더하는 기여가 뚜렷하지 않습니다. 다음 함수는 체중과 특정 부리 길이를 받아, 나머지 측정값을 평균에 고정한 상태로 수컷 확률을 계산하고 그립니다.

In [60]:
def plot_predictions(mass_range, culmen_length, **options):
    """Plot predicted probabilities as a function of mass."""
    df = pd.DataFrame({"mass": mass_range})
    df["flipper_length"] = adelie["flipper_length"].mean()
    df["culmen_length"] = culmen_length
    df["culmen_depth"] = adelie["culmen_depth"].mean()
    fit_ys = result.predict(df)
    plt.plot(mass_range, fit_ys, **options)

부리 길이를 평균보다 한 표준편차 작게, 평균으로, 한 표준편차 크게 놓고 비교합니다.

In [61]:
culmen_length = adelie["culmen_length"]
m, s = culmen_length.mean(), culmen_length.std()

In [62]:
plot_predictions(mass_range, m + s, ls="--", label="Above average culmen length")
plot_predictions(mass_range, m, alpha=0.5, label="Average culmen length")
plot_predictions(mass_range, m - s, ls=":", label="Below average culmen length")

decorate(xlabel="Mass (g)", ylabel="Prob(male)")

체중이 클수록, 그리고 같은 체중에서 부리가 길수록 수컷일 예측 확률이 높습니다. 이 분석은 원래 연구의 방법과 비슷합니다. 그 연구는 암수의 몸이 얼마나 다른지 나타내는 **성적 이형성**을 다룹니다. 몸의 측정값으로 성별을 더 정확히 구별할 수 있다면 이형성이 더 큰 것으로 볼 수 있습니다.

원래 논문은 Gorman, Williams, Fraser(2014), 「Ecological sexual dimorphism and environmental variability within a community of Antarctic penguins」, *PLOS ONE* 9(3), e90081, [DOI: 10.1371/journal.pone.0090081](https://doi.org/10.1371/journal.pone.0090081)입니다.

다른 종에서도 같은 방법을 써 보겠습니다. 자료에는 젠투펭귄 119마리의 유효 체중·성별 측정값도 있습니다. 다음 함수로 종을 선택합니다.

In [ ]:
def get_species(penguins, species):
    df = penguins.query(f'Species.str.startswith("{species}")').copy()
    df = df.loc[df["Sex"].isin(["MALE", "FEMALE"])].copy()
    df["y"] = (df["Sex"] == "MALE").astype(int)
    return df

In [64]:
gentoo = get_species(penguins, "Gentoo")
len(gentoo)

젠투펭귄의 로지스틱 회귀 결과입니다.

In [65]:
formula = "y ~ mass + flipper_length + culmen_length + culmen_depth"
model = smf.logit(formula, data=gentoo)
result = model.fit(disp=False)
display_summary(result)

유사 결정계수는 약 0.850로 아델리펭귄 모형의 약 0.735보다 큽니다. 원저자는 이 결과를 젠투펭귄의 성적 이형성이 더 크다는 단서로 해석합니다.

> 번역자 주: 유사 결정계수는 설명된 분산의 비율이 아닙니다. 특히 서로 다른 종·표본 사이의 값만으로 분류 정확도의 우열이 확정되지는 않습니다. 실제 분류 성능은 별도 자료와 같은 평가 기준으로 확인해야 합니다. 로지스틱 예측도 관측한 체중 범위와 자료의 성별 구성에 의존합니다.

## 용어 정리

- **회귀**: 반응변수와 설명변수 사이의 관계를 모형으로 나타내고 계수를 추정하는 방법.
- **반응변수**: 모형으로 예측하려는 변수. 종속변수라고도 합니다.
- **설명변수**: 예측에 사용하는 변수. 독립변수라고도 하지만 서로 독립이라는 뜻은 아닙니다.
- **단순회귀**: 반응변수와 설명변수가 하나씩인 회귀.
- **다중회귀**: 반응변수는 하나이고 설명변수가 여러 개인 회귀.
- **계수**: 절편과 각 설명변수에 대응하는 추정 기울기.
- **범주형 변수**: 측정 크기보다 구별되는 집단을 나타내는 변수.
- **통제변수**: 관심 관계를 다른 변수의 관계와 함께 조정하기 위해 포함한 변수.
- **일반화선형모형**: 반응변수의 분포와 연결함수를 지정하는 회귀 모형들.
- **로지스틱 회귀**: 반응변수가 두 범주일 때 사용하는 일반화선형모형.

## 연습문제

### 연습문제 11.1

남아는 여아보다 무거운가요? NSFG에서 체중을 반응변수, `babysex`를 범주형 설명변수로 회귀하세요. 1은 남아, 2는 여아입니다. 추정 차이와 p값은 얼마인가요? 산모 나이를 통제하면 차이가 얼마나 달라지나요?

In [66]:
# Solution goes here

In [67]:
# Solution goes here

In [68]:
# Solution goes here

### 연습문제 11.2

트리버스–윌러드 가설은 포유류의 성비가 산모의 나이·크기·건강·사회적 지위 같은 ‘모체 상태’에 의존할 수 있다고 제안합니다. 인간 연구도 있지만 결과는 일치하지 않습니다.

배경은 [트리버스–윌러드 가설 설명](https://en.wikipedia.org/wiki/Trivers-Willard_hypothesis)을 참고하세요.

산모 나이와 남아 출산 확률 사이에 관계가 있는지 살펴보세요. 아기 성별을 반응변수로 로지스틱 회귀를 하세요. 나이가 많은 산모의 남아 출산 확률은 더 높은가요, 낮은가요? 나이의 이차항을 추가하면 어떻게 되나요?

로지스틱 회귀에 쓰기 위해 남아는 1, 여아는 0으로 재부호화합니다.

In [69]:
valid["y"] = (valid["babysex"] == 1).astype(int)

In [70]:
# Solution goes here

In [71]:
# Solution goes here

### 연습문제 11.3

아델리펭귄의 체중을 `flipper_length`, `culmen_depth`, 범주형 `Sex`로 회귀하세요. 날개 길이와 부리 깊이를 통제하면 수컷이 얼마나 더 무거운가요? 부리 깊이를 평균으로 놓고, 암수 각각의 날개 길이별 예측 체중을 그리세요.

In [72]:
# Solution goes here

In [73]:
# Solution goes here

In [74]:
# Solution goes here

### 연습문제 11.4

`get_species`로 턱끈펭귄을 선택하고, 네 측정값으로 성별을 예측하는 로지스틱 회귀를 하세요. 유사 결정계수를 다른 두 종과 비교하세요. 이 비교를 이형성으로 해석할 때의 한계도 생각해 보세요.

In [75]:
chinstrap = get_species(penguins, "Chinstrap")
len(chinstrap)

In [76]:
# Solution goes here

In [77]:
# Solution goes here

[Think Stats: Exploratory Data Analysis in Python, 3rd Edition](https://allendowney.github.io/ThinkStats/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

코드 라이선스: [MIT](https://mit-license.org/)

본문·그림과 한국어 번역 라이선스: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)